# Gridsearch_CV

# 🔍 Grid Search with Cross-Validation — Exhaustive Hyperparameter Tuning

**Grid Search** is a brute-force strategy for hyperparameter optimization.  
It systematically explores all combinations of predefined hyperparameter values to find the best-performing configuration.


## ⚙️ How It Works

1. **Define a grid** of possible hyperparameter values to test.  
   Example:  
   - `max_depth`: [3, 5, 7]  
   - `learning_rate`: [0.01, 0.1, 0.2]

2. For **each combination** in the grid:
   - Train the model using **cross-validation** (e.g., 5-fold CV).
   - Compute the **average validation score** across folds.

3. **Select the combination** that yields the **highest average score**.

## ✅ Advantage

- **Exhaustive and reliable**:  
  Guarantees the best result **within the defined grid**.  
  Ideal when the search space is small and interpretability matters.


## ❌ Disadvantage

- **Computationally expensive**:  
  The number of combinations grows **exponentially** with the number of hyperparameters.  
  Can be **very slow** for large grids or complex models.

## 📌 When to Use Grid Search

- When you have **few hyperparameters** and a **small search space**.  
- When you want **guaranteed optimality** within the grid.  
- When interpretability and reproducibility are important.


In [137]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('usa_houses.csv')
df = df[(df['price'] < 2700000) & (df['price'] > 10000)]

#df = df[(df["price"] < 1000000) & (df["price"] > 10000)]
x = df[["sqft_living", "bedrooms", "bathrooms", "floors"]]
y = df["price"]


X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42
)

# Training The Model 

In [138]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge

pipeline = Pipeline([
    ("poly", PolynomialFeatures()),
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])



# Gridsearch and Fitting 

In [139]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "poly__degree": [1, 2, 3, 4,5,6],
    "ridge__alpha": [0.1, 0.2, 0.3, 0.4, 0.5, 1, 2, 4]  # [0.001, 0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(pipeline,param_grid=param_grid, scoring="r2" , cv=3)
grid_search.fit(X_train, y_train)


,estimator,"Pipeline(step...e', Ridge())])"
,param_grid,"{'poly__degree': [1, 2, ...], 'ridge__alpha': [0.1, 0.2, ...]}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,degree,2


# Evaluating Model 

In [140]:
from sklearn.metrics import root_mean_squared_error, r2_score


# Best model and evaluation
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)


print("📊 Best Parameters:", grid_search.best_params_)
print("📈 Best R² Score:", round(grid_search.best_score_, 3))

📊 Best Parameters: {'poly__degree': 2, 'ridge__alpha': 4}
📈 Best R² Score: 0.483


# 🎲 Randomized Search with Cross-Validation — Efficient Hyperparameter Exploration

When the number of hyperparameters or the size of their value ranges becomes large, **Grid Search** can quickly become impractical.  
**Randomized Search** offers a powerful alternative: it explores the hyperparameter space **randomly but efficiently**, without exhaustively testing every combination.

## ⚙️ How It Works

1. **Define a range or distribution** for each hyperparameter.  
   Example:  
   - `alpha`: from 0.01 to 10  
   - `degree`: from 1 to 5

2. **Randomly sample `n` combinations** from these ranges.  
   These combinations are selected independently and uniformly (or from specified distributions).

3. For each sampled combination:
   - Train the model using **cross-validation** (e.g., 5-fold CV).
   - Compute the **average validation score**.

4. **Select the combination** with the **best average performance**.

## ✅ Advantages

- **Much faster** than Grid Search, especially in **high-dimensional spaces**.
- Can discover **near-optimal configurations** with surprisingly few iterations.
- Works well when hyperparameters are **continuous or have large ranges**.
- Easy to parallelize and scale.

## ❌ Disadvantages

- **No guarantee** of finding the absolute best combination.
- Performance depends on the **number of iterations** — too few may miss good regions.
- Results can vary between runs unless a **random seed** is fixed.

## 📌 When to Use Randomized Search

- When your search space is **large or continuous**.
- When you want **quick tuning** without exhaustive search.
- When you're working with **complex models** (e.g., ensembles, deep learning).
- When you want to **combine speed and flexibility**.

👉 Randomized Search is like throwing darts at the hyperparameter space —  
you may not hit the bullseye, but you’ll often land **close enough** to win the game.

# Model Training Same datas


In [141]:
from sklearn.model_selection import RandomizedSearchCV 
from scipy.stats  import stats

# 🔗 Pipeline: scaling + ElasticNet
model = Pipeline([
    ("poly", PolynomialFeatures()),   
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])



# Gridsearch

In [ ]:
from scipy.stats  import stats
# 🎯 Parameter distributions
param_dist = {
    "poly__degree": [1, 2, 3,4,5,6],
    "ridge__alpha": spst.loguniform(1e-3, 1e2),   
}

# 🔍 Randomized search setup
rand_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring="r2",
    random_state=42,
    verbose=1
)


# Train and Evaluation 

In [143]:
# 🚀 Fit the search
rand_search.fit(X_train, y_train)

# 📈 Evaluate best model
best_model = rand_search.best_estimator_
y_pred = best_model.predict(X_test)

print("📊 Best Parameters:", rand_search.best_params_)
print("📈 Best R² Score:", round(rand_search.best_score_, 3))

Fitting 3 folds for each of 10 candidates, totalling 30 fits
📊 Best Parameters: {'poly__degree': 2, 'ridge__alpha': np.float64(4.073745196058386)}
📈 Best R² Score: 0.483
